# Attention Lab 5 · Soft vs Hard Attention

> **Types of Attention — Lab 5.** Run top to bottom (Runtime → Run all). Read the plain-English notes, run the code,
> then do the **Your turn 🧪** cell. Everything is built from scratch with NumPy so nothing is hidden.

### In plain English
When a word focuses, does it **gently split** its attention or **commit to one thing**?

- **Soft attention = a dimmer switch.** Focus is split across everything by percentages (70%, 20%, 10%) and the result
  is a blend. Because it's smooth, the model learns easily — which is why nearly every real system uses it.
- **Hard attention = an on/off switch.** Pick exactly one word. Cheaper and easy to explain (you can point to the one
  thing it looked at) — but "pick one" is an abrupt choice with no useful gradient, so it's hard to train and stays
  mostly a research idea.

**Temperature** is the dial between them: low temperature sharpens soft attention until it almost picks one thing.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
VIOLET, ROSE = "#6C5CE7", "#FB7185"
def softmax(x):
    x = x - x.max(); e = np.exp(x); return e / e.sum()

keys   = ["k1","k2","k3","k4","k5","k6"]
scores = np.array([0.4, 1.8, 0.9, 2.4, 0.6, 1.1])
values = np.array([10., 20., 30., 40., 50., 60.])   # what each key hands over

## 1 · Soft attention blends; hard attention picks

In [ ]:
def soft(scores, values, T=1.0):
    w = softmax(scores / T)
    return w @ values, w

def hard(scores, values):
    w = np.zeros_like(scores); w[scores.argmax()] = 1.0     # pick the best, ignore the rest
    return w @ values, w

out_s, w_s = soft(scores, values)
out_h, w_h = hard(scores, values)
print("soft weights:", np.round(w_s,3), "-> output", round(out_s,2))
print("hard weights:", w_h.astype(int),  "-> output", round(out_h,2), f"(just k{scores.argmax()+1})")

## 2 · Temperature: the dial from soft to hard

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(13,3), sharey=True)
for ax, T in zip(axes, [3.0, 1.0, 0.3, 0.05]):
    _, w = soft(scores, values, T)
    ax.bar(keys, w, color=VIOLET); ax.set_title(f"T = {T}"); ax.set_ylim(0,1)
axes[0].set_ylabel("attention weight")
fig.suptitle("low temperature -> soft attention approaches hard attention", y=1.06)
plt.tight_layout(); plt.show()
for T in [3.0, 1.0, 0.3, 0.05]:
    _, w = soft(scores, values, T)
    print(f"T={T:<5} max weight {w.max():.3f}  effective #keys {1/(w**2).sum():.2f}")

## 3 · Why hard attention is hard to train
Soft attention's output changes **smoothly** when a score changes, so we can follow the gradient. Hard attention's
output **jumps** — the derivative is zero almost everywhere and undefined at the jump.

In [ ]:
eps = 1e-5
def out_soft(s2): 
    sc = scores.copy(); sc[1] = s2; return soft(sc, values)[0]
def out_hard(s2):
    sc = scores.copy(); sc[1] = s2; return hard(sc, values)[0]

s2 = 1.8
g_soft = (out_soft(s2+eps) - out_soft(s2-eps)) / (2*eps)
g_hard = (out_hard(s2+eps) - out_hard(s2-eps)) / (2*eps)
print(f"gradient of output w.r.t. score of k2:")
print(f"   soft: {g_soft:.4f}   <- useful learning signal")
print(f"   hard: {g_hard:.4f}   <- zero: nothing to learn from")

xs = np.linspace(0, 4, 400)
plt.figure(figsize=(6.5,3.4))
plt.plot(xs, [out_soft(x) for x in xs], color=VIOLET, lw=2, label="soft (smooth)")
plt.plot(xs, [out_hard(x) for x in xs], color=ROSE, lw=2, label="hard (jumps)")
plt.xlabel("score of k2"); plt.ylabel("attention output"); plt.legend()
plt.title("soft is differentiable; hard is a step function"); plt.tight_layout(); plt.show()

## Your turn 🧪
1. At which temperature does soft attention put >99% on one key? (Sweep `T` down.)
2. Implement **stochastic hard attention**: sample a key from the softmax distribution instead of taking the argmax.
   Average the output over 1000 samples — does it approach the soft output?
3. Why might hard attention be preferable when you must *explain* a model's decision?

In [ ]:
# Your turn: stochastic hard attention converges to soft attention on average
rng = np.random.default_rng(0)
p = softmax(scores)
samples = rng.choice(len(keys), size=1000, p=p)
print("mean of sampled hard outputs:", round(values[samples].mean(), 2))
print("soft attention output       :", round(out_s, 2))